Here is a **clean, industry-grade approach** to build an **Order-Support Agent (Order Chart Bot)** using:

* **LangGraph**
* **Open LLM models** (OpenAI, Llama, Mistral, etc.)
* **Vector DB** (Chroma / Weaviate / Pinecone)
* **Your CSV file (order.csv / adidas.csv)**

I’ll give you the **architecture**, **pipeline**, and **code template** you can directly use.

---

# ✅ **1. High-Level Architecture**

```
order.csv  →  Preprocessing  →  Vector DB (Embeddings)
                                      ↑
                                      |
User Query → LangGraph → Routing →   RAG → LLM → Final Answer
```

Your agent will support questions like:
✔ "Show orders returned last month"
✔ "Which customers returned most products?"
✔ "Summaries, charts, insights"
✔ "Explain return reasons"

---

# ✅ **2. Recommended Pipeline**

### **Step 1 — Load & Clean CSV**

Use pandas to load your order CSV or adidas.csv.

```python
import pandas as pd

df = pd.read_csv("adidas.csv")
```

---

### **Step 2 — Convert Rows to Documents**

Each row becomes a document for vector search.

```python
docs = []
for i, row in df.iterrows():
    text = "\n".join([f"{col}: {row[col]}" for col in df.columns])
    docs.append({"id": str(i), "text": text})
```

---

### **Step 3 — Store in Vector DB**

Choose one:

* **Chroma (local, free)**
* **Pinecone (cloud)**
* **Weaviate**

Example (Chroma):

```python
from chromadb import Client
client = Client()

collection = client.create_collection("orders")

for d in docs:
    collection.add(documents=[d["text"]], ids=[d["id"]])
```

---

# ✅ **3. Build LangGraph Agent**

### **Agent Nodes**

You need 3 main nodes:

### **1️⃣ Query Understanding Node**

Classifies intent (summary, visualization, raw data, insights, returns, etc.)

### **2️⃣ Retrieval Node (Vector DB Search)**

Fetches relevant docs.

```python
def retrieve(state):
    query = state["query"]
    results = collection.query(query_texts=[query], n_results=5)
    return {"context": results["documents"]}
```

### **3️⃣ LLM Response Node**

Does RAG generation using Open LLM.

```python
from openai import OpenAI
client = OpenAI()

def llm_answer(state):
    prompt = f"""
    You are an Order-Insight Agent.

    User question:
    {state["query"]}

    Relevant order records:
    {state["context"]}

    Provide accurate answers with summary, explanation,
    and calculations when needed.
    """

    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}]
    )

    return {"answer": response.choices[0].message["content"]}
```

---

# ✅ **4. Build LangGraph Wiring**

```python
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    query: str
    context: list
    answer: str

graph = StateGraph(AgentState)

graph.add_node("retrieve", retrieve)
graph.add_node("llm_answer", llm_answer)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "llm_answer")
graph.add_edge("llm_answer", END)

agent = graph.compile()
```

---

# ✅ **5. Run the Agent**

```python
result = agent.invoke({"query": "Which products had highest returns?"})
print(result["answer"])
```

---

# ✅ **6. (Optional) Generate Charts**

You can add a **Chart Generator Node**:

* Accepts user query
* Detects need for a chart ("plot", "graph", "trend", etc.)
* Uses matplotlib to generate image
* Returns the file path to LLM

This node can be inserted between `retrieve → llm_answer`.

---

# 🎯 **Final Capabilities Your Agent Will Have**

✔ Q&A about orders and returns
✔ Trends and charts (monthly returns, region-wise, product-wise)
✔ Customer-level insights
✔ RAG-based answers using CSV data
✔ Structured calculations (totals, averages, % return rate)
✔ Works with **any CSV dataset**

---

# If you want, I can generate:

✅ Complete project folder structure
✅ Full code for LangGraph agent
✅ Docker version
✅ FastAPI endpoint to expose the bot
✅ UI (Streamlit or React)

Would you like the **full runnable code project**?


# Order Chart Bot — Complete Project

This repository contains a complete, runnable project to build an Order Chart Bot using **LangGraph**, an **Open LLM**, and a **vector database (Chroma)**. It includes a Docker setup, a FastAPI backend exposing the agent, and a Streamlit UI.

> **CSV path used by default**: `/mnt/data/adidas.csv`

---

## Project structure

```
order-chart-bot/
├── README.md
├── requirements.txt
├── Dockerfile
├── docker-compose.yml
├── .env.example
├── src/
│   ├── __init__.py
│   ├── main.py                # FastAPI app
│   ├── langgraph_agent.py     # LangGraph graph and nodes
│   ├── vector_store.py        # Chroma wrapper
│   ├── embeddings.py         # wrapper for embeddings
│   ├── utils.py               # helpers (chart generation, prompts)
│   └── streamlit_app.py      # Streamlit UI
├── tests/
│   └── test_agent.py
└── deployment/
    └── nginx.conf
```

---

## README (high level usage)

See the code inside `src/` for implementation details. Quick start:

1. Copy `.env.example` to `.env` and set your LLM credentials (e.g. `OPENAI_API_KEY`) or configure local Open LLM settings.
2. Build and run with Docker Compose:

```bash
docker compose up --build
```

This starts:

* FastAPI at `http://localhost:8000`
* Streamlit UI at `http://localhost:8501`

---

## Key implementation notes

* The code reads the provided CSV at `/mnt/data/adidas.csv` by default. Streamlit UI allows uploading another CSV.
* We convert each CSV row to a text document and store in **Chroma**.
* LangGraph routes queries: intent detection -> semantic retrieval -> LLM (RAG) -> optional chart generation.
* The FastAPI endpoint `/query` accepts `POST` with JSON `{ "query": "...", "n_results": 5 }` and returns a JSON answer and optional chart URL.

---

## requirements.txt

```
fastapi
uvicorn[standard]
langgraph
chromadb
openai
python-dotenv
pandas
numpy
scikit-learn
sentence-transformers
matplotlib
streamlit
requests
```

(If you use a local LLM provider or `llama-cpp-python`, add those packages accordingly.)

---

## .env.example

```
OPENAI_API_KEY=sk-...
CHROMA_DIR=./chroma_db
HOST=0.0.0.0
PORT=8000
```

---

## src/vector_store.py

```python
# src/vector_store.py
from chromadb import Client
from chromadb.config import Settings
import os
from typing import List

class ChromaStore:
    def __init__(self, persist_directory: str | None = None):
        settings = Settings(chroma_db_impl="duckdb+parquet")
        self.client = Client(settings)
        self.collection = None
        self.persist_directory = persist_directory
        self.collection_name = "orders"

    def create_or_get_collection(self):
        try:
            self.collection = self.client.get_collection(self.collection_name)
        except Exception:
            self.collection = self.client.create_collection(self.collection_name)
        return self.collection

    def add_documents(self, ids: List[str], texts: List[str], metadatas: List[dict] | None = None):
        if self.collection is None:
            self.create_or_get_collection()
        self.collection.add(ids=ids, documents=texts, metadatas=metadatas)

    def query(self, query_text: str, n_results: int = 5):
        if self.collection is None:
            self.create_or_get_collection()
        res = self.collection.query(query_texts=[query_text], n_results=n_results)
        # res contains 'ids', 'documents', 'distances'
        return res
```

---

## src/embeddings.py

```python
# src/embeddings.py
from typing import List
import os
from openai import OpenAI

# Simple wrapper that uses OpenAI embeddings. Swap with whichever embeddings provider you prefer.

class EmbeddingsProvider:
    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = OpenAI()
        self.model = model

    def embed(self, texts: List[str]):
        out = self.client.embeddings.create(input=texts, model=self.model)
        return [d.embedding for d in out.data]
```

---

## src/langgraph_agent.py

```python
# src/langgraph_agent.py
from langgraph.graph import StateGraph, END
from typing import TypedDict
from .vector_store import ChromaStore
from .embeddings import EmbeddingsProvider
from .utils import build_prompt
import os

class AgentState(TypedDict):
    query: str
    context: list
    answer: str

# Nodes

store = ChromaStore(persist_directory=os.getenv("CHROMA_DIR", "./chroma_db"))
embedder = EmbeddingsProvider()


def intent_node(state: AgentState):
    # Very small heuristic — can be replaced with a proper intent classifier.
    q = state["query"].lower()
    want_chart = any(k in q for k in ["plot", "chart", "graph", "trend"])
    return {"want_chart": want_chart}


def retrieve_node(state: AgentState, n_results: int = 5):
    query = state["query"]
    res = store.query(query, n_results=n_results)
    documents = []
    for doc in res.get("documents", []):
        documents.append(doc)
    state["context"] = documents
    return {"context": documents}


def llm_node(state: AgentState):
    # This builds a RAG prompt and calls an LLM (OpenAI in this example). Swap for Open LLM provider if needed.
    prompt = build_prompt(state["query"], state.get("context", []))

    # call OpenAI ChatCompletion
    from openai import OpenAI
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800
    )
    answer = response.choices[0].message.content
    state["answer"] = answer
    return {"answer": answer}


def make_graph():
    graph = StateGraph(AgentState)
    graph.add_node("intent", intent_node)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("llm", llm_node)

    graph.set_entry_point("intent")
    graph.add_edge("intent", "retrieve")
    graph.add_edge("retrieve", "llm")
    graph.add_edge("llm", END)
    return graph


# convenience
agent = make_graph()
```

---

## src/utils.py

```python
# src/utils.py
from typing import List


def row_list_to_text(rows: List[dict]):
    texts = []
    for r in rows:
        lines = [f"{k}: {v}" for k, v in r.items()]
        texts.append("\n".join(lines))
    return texts


def build_prompt(query: str, context: List[str]):
    context_text = "\n\n---\n\n".join(context[:10])
    prompt = f"You are an Order Insight assistant. Use the following order records to answer the question.\n\nQuestion:\n{query}\n\nRelevant records:\n{context_text}\n\nProvide an accurate, concise answer and explain any calculations. If a chart is requested, say 'chart:yes' and provide instructions to generate it."
    return prompt


def generate_chart(df, chart_type: str = "bar", out_path: str = "/tmp/chart.png"):
    import matplotlib.pyplot as plt
    import pandas as pd

    if chart_type == "returns_by_month":
        # expects df with 'order_date' and 'returned' flag
        df['order_date'] = pd.to_datetime(df['order_date'])
        df['month'] = df['order_date'].dt.to_period('M')
        agg = df.groupby('month')['returned'].sum()
        agg.plot(kind='bar')
        plt.tight_layout()
        plt.savefig(out_path)
        return out_path

    # fallback
    plt.figure()
    plt.text(0.5, 0.5, 'No chart generated', ha='center')
    plt.savefig(out_path)
    return out_path
```

---

## src/main.py (FastAPI)

```python
# src/main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from .langgraph_agent import agent
from .vector_store import ChromaStore
import os
import pandas as pd
from .utils import generate_chart

app = FastAPI()

DATA_PATH = os.getenv("DATA_PATH", "/mnt/data/adidas.csv")

class QueryIn(BaseModel):
    query: str
    n_results: int = 5

@app.on_event("startup")
async def startup_event():
    # load CSV and seed vector store if empty
    global store
    store = ChromaStore(persist_directory=os.getenv("CHROMA_DIR", "./chroma_db"))
    store.create_or_get_collection()

    # If collection empty, seed from CSV
    # Simple check: if no documents, add them
    try:
        df = pd.read_csv(DATA_PATH)
    except Exception as e:
        print(f"Failed to read {DATA_PATH}: {e}")
        df = None

    if df is not None:
        # create docs
        ids = []
        texts = []
        metadatas = []
        for i, row in df.iterrows():
            ids.append(str(i))
            text = "\n".join([f"{c}: {row[c]}" for c in df.columns])
            texts.append(text)
            metadatas.append({"index": i})
        # Add to store. In production, dedupe or check before adding.
        store.add_documents(ids, texts, metadatas)

@app.post("/query")
async def query_endpoint(q: QueryIn):
    try:
        state = {"query": q.query}
        # run the graph synchronously (LangGraph graph invocation depends on library API)
        result = agent.invoke(state)
        answer = result.get("answer") or state.get("answer")

        # if want chart keyword detected, generate chart and return path
        chart_path = None
        if "chart:yes" in (answer or ""):
            # naive example generating returns_by_month
            df = pd.read_csv(DATA_PATH)
            chart_path = generate_chart(df, chart_type="returns_by_month", out_path="/tmp/chart.png")

        return {"answer": answer, "chart": chart_path}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
```

---

## src/streamlit_app.py

```python
# src/streamlit_app.py
import streamlit as st
import requests
import pandas as pd
import os

BACKEND = os.getenv("BACKEND_URL", "http://localhost:8000")
DEFAULT_CSV = "/mnt/data/adidas.csv"

st.title("Order Chart Bot — Streamlit UI")

uploaded = st.file_uploader("Upload orders CSV (optional)")
if uploaded is not None:
    df = pd.read_csv(uploaded)
    st.session_state['df_path'] = None
else:
    try:
        df = pd.read_csv(DEFAULT_CSV)
    except Exception:
        df = None

if df is not None:
    st.write("Preview of orders CSV:")
    st.dataframe(df.head())

query = st.text_input("Ask a question about orders and returns:")
if st.button("Ask") and query:
    with st.spinner("Querying agent..."):
        resp = requests.post(f"{BACKEND}/query", json={"query": query, "n_results": 5})
        if resp.status_code == 200:
            data = resp.json()
            st.subheader("Answer")
            st.write(data.get('answer'))
            chart = data.get('chart')
            if chart:
                st.image(chart)
        else:
            st.error(f"Error: {resp.text}")
```

---

## Dockerfile

```
FROM python:3.11-slim
WORKDIR /app
COPY ./requirements.txt /app/requirements.txt
RUN apt-get update && apt-get install -y build-essential libglib2.0-0 && rm -rf /var/lib/apt/lists/*
RUN pip install --no-cache-dir -r /app/requirements.txt
COPY ./src /app/src
ENV PYTHONPATH=/app
EXPOSE 8000 8501
CMD ["uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

---

## docker-compose.yml

```yaml
version: '3.8'
services:
  backend:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - CHROMA_DIR=/data/chroma
      - DATA_PATH=/mnt/data/adidas.csv
    volumes:
      - ./data:/data
      - /mnt/data:/mnt/data:ro
  streamlit:
    image: python:3.11-slim
    command: bash -lc "pip install -r /app/requirements.txt && streamlit run /app/src/streamlit_app.py --server.port 8501 --server.address 0.0.0.0"
    ports:
      - "8501:8501"
    volumes:
      - ./:/app
    environment:
      - BACKEND_URL=http://backend:8000
      - DATA_PATH=/mnt/data/adidas.csv
```

---

## Notes and next steps

* Replace OpenAI usage with your selected *Open LLM* provider if you want to host models locally.
* Improve intent classification and add a conversation memory node if you want multi-turn.
* Add proper vector indexing/upsert checks to avoid duplicating documents on each start.

---

## Tests

A simple pytest can call the FastAPI endpoint with example questions in `tests/test_agent.py`.

---

If you'd like, I can also:

* Export this project as a downloadable zip.
* Generate a ready-to-run `docker compose up` command with more production-ready settings.
* Swap vector store to `Weaviate` or `Pinecone` if you prefer cloud hosting.
